# Study 1: GI VQA Baseline and Visual Faithfulness Audit

Fresh-start notebook for the first PhD study: move from the original proof-of-concept notebook to a reproducible full-dataset pipeline and evaluation study.

**Study aim:** evaluate whether a fine-tuned VLM for GI endoscopy VQA produces answers and visual explanations that are faithful to clinically relevant image evidence.

**Core hypothesis:** if a visual explanation is faithful, removing the most salient regions should degrade answer correctness or confidence more than removing random regions of the same size.

## Study Design

### Research Questions

1. **VQA performance:** How well does a fine-tuned VLM answer GI endoscopy VQA questions on the full Kvasir-VQA-x1 test split?
2. **Visual faithfulness:** Do attention or saliency maps identify image regions that causally affect the generated answer?
3. **Confidence and failure modes:** Are incorrect or visually ungrounded answers associated with misleading confidence or plausible-looking explanations?

### Model Conditions

- Zero-shot/base `google/paligemma-3b-pt-224`
- QLoRA fine-tuned `google/paligemma-3b-pt-224` on the full Kvasir-VQA-x1 training split
- Optional later ablations: LoRA rank, number of epochs, frozen vs unfrozen vision encoder, alternative VLM

### Primary Endpoint

Difference between targeted-deletion degradation and random-deletion degradation.

For each test item:

```text
original image + question
  -> answer, confidence, explanation map
  -> mask top-k salient regions
  -> re-run answer/confidence
  -> compare with random and least-salient controls
```

## 0. Environment and Reproducibility

This notebook deliberately avoids hardcoded Hugging Face or Weights & Biases tokens. Set them outside the notebook if needed:

```bash
export HF_TOKEN=...
export WANDB_API_KEY=...
```

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime
import json
import os
import random
import statistics

SEED = 42
random.seed(SEED)

@dataclass
class StudyConfig:
    model_name: str = "google/paligemma-3b-pt-224"
    data_dir: Path = Path("Kvasir-VQA-x1")
    image_dir: Path = Path("Kvasir-VQA-x1/images")
    output_dir: Path = Path("study_outputs/study_1_gi_vqa_faithfulness")
    train_jsonl: Path = Path("Kvasir-VQA-x1/Kvasir-VQA-x1-train.jsonl")
    test_jsonl: Path = Path("Kvasir-VQA-x1/Kvasir-VQA-x1-test.jsonl")
    dev_jsonl: Path = Path("Kvasir-VQA-x1/Kvasir-VQA-x1-dev-1000-seed42.jsonl")
    val_sample_size: int = 1000
    inference_sample_size: int | None = None  # None means full test set
    deletion_levels: tuple[float, ...] = (0.05, 0.10, 0.20, 0.30)
    random_deletion_repeats: int = 3

cfg = StudyConfig()
cfg.data_dir.mkdir(parents=True, exist_ok=True)
cfg.image_dir.mkdir(parents=True, exist_ok=True)
cfg.output_dir.mkdir(parents=True, exist_ok=True)

print("Study output dir:", cfg.output_dir)
print("HF token available:", bool(os.getenv("HF_TOKEN")))
print("W&B key available:", bool(os.getenv("WANDB_API_KEY")))

## 1. Data Preparation

Create full train/test JSONL files for `ms-swift`, cache one image per `img_id`, and create a deterministic development validation subset for fast iteration.

The final study should report on the full official test split, not the development subset.

In [ ]:
# Run once in a fresh environment.
# !pip install datasets tqdm pillow

from datasets import load_dataset
from tqdm.auto import tqdm

def write_jsonl(split: str, out_path: Path) -> Path:
    ds = load_dataset("SimulaMet/Kvasir-VQA-x1", split=split)
    with out_path.open("w", encoding="utf-8") as f:
        for row in ds:
            record = {
                "messages": [
                    {"role": "user", "content": f"<image>{row['question']}"},
                    {"role": "assistant", "content": row["answer"]},
                ],
                "images": [str(cfg.image_dir / f"{row['img_id']}.jpg")],
                "metadata": {
                    "img_id": row["img_id"],
                    "split": split,
                },
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    return out_path

def cache_images() -> None:
    host = load_dataset("SimulaMet-HOST/Kvasir-VQA", split="raw")
    seen = set()
    for idx, row in tqdm(enumerate(host), total=len(host), desc="Caching unique images"):
        img_id = row["img_id"]
        if img_id in seen:
            continue
        seen.add(img_id)
        out_path = cfg.image_dir / f"{img_id}.jpg"
        if not out_path.exists():
            row["image"].save(out_path)

def make_dev_subset(source_path: Path, out_path: Path, n: int, seed: int) -> Path:
    rows = source_path.read_text(encoding="utf-8").splitlines()
    rng = random.Random(seed)
    selected = rng.sample(rows, min(n, len(rows)))
    out_path.write_text("\n".join(selected) + "\n", encoding="utf-8")
    return out_path

if not cfg.train_jsonl.exists() or not cfg.test_jsonl.exists():
    cache_images()
    write_jsonl("train", cfg.train_jsonl)
    write_jsonl("test", cfg.test_jsonl)

if not cfg.dev_jsonl.exists():
    make_dev_subset(cfg.test_jsonl, cfg.dev_jsonl, cfg.val_sample_size, SEED)

print("Train:", cfg.train_jsonl)
print("Test:", cfg.test_jsonl)
print("Dev:", cfg.dev_jsonl)

## 2. Data Audit

Before training, verify counts, missing images, duplicate IDs, and a few representative samples. This becomes part of the reproducibility appendix.

In [ ]:
def iter_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def audit_jsonl(path: Path) -> dict:
    total = 0
    missing_images = []
    img_ids = []
    answer_lengths = []
    for record in iter_jsonl(path):
        total += 1
        img_path = Path(record["images"][0])
        if not img_path.exists():
            missing_images.append(str(img_path))
        img_ids.append(record.get("metadata", {}).get("img_id"))
        answer = record["messages"][1]["content"]
        answer_lengths.append(len(answer.split()))
    return {
        "path": str(path),
        "total": total,
        "unique_img_ids": len(set(img_ids)),
        "missing_images": len(missing_images),
        "example_missing_images": missing_images[:5],
        "mean_answer_words": statistics.mean(answer_lengths) if answer_lengths else 0,
    }

for path in [cfg.train_jsonl, cfg.test_jsonl, cfg.dev_jsonl]:
    print(json.dumps(audit_jsonl(path), indent=2))

sample = next(iter_jsonl(cfg.train_jsonl))
print("\nExample question:", sample["messages"][0]["content"].replace("<image>", ""))
print("Example answer:", sample["messages"][1]["content"])
print("Example image:", sample["images"][0])

## 3. Full-Dataset Fine-Tuning

Use the full training split for the study model. Use the deterministic dev subset for fast checkpoint selection while prototyping. For final reporting, run inference on the full test split.

Set `DRY_RUN = False` only when the environment has the required GPU, packages, and credentials.

In [ ]:
# Run once in a GPU environment.
# !pip install ms-swift bitsandbytes wandb

import shlex
import subprocess

RUN_ID = datetime.now().strftime("%y%m%d-%H%M")
HUB_MODEL_ID = f"Kvasir-VQA-x1-study1-paligemma-lora-{RUN_ID}"
TRAIN_OUTPUT_DIR = cfg.output_dir / "training" / HUB_MODEL_ID

swift_cmd = [
    "swift", "sft",
    "--dataset", str(cfg.train_jsonl),
    "--val_dataset", str(cfg.dev_jsonl),
    "--model", cfg.model_name,
    "--max_length", "512",
    "--train_type", "lora",
    "--torch_dtype", "float16",
    "--quant_method", "bnb",
    "--quant_bits", "4",
    "--bnb_4bit_compute_dtype", "float16",
    "--bnb_4bit_quant_type", "nf4",
    "--bnb_4bit_use_double_quant", "true",
    "--num_train_epochs", "1",
    "--per_device_train_batch_size", "4",
    "--per_device_eval_batch_size", "4",
    "--gradient_accumulation_steps", "4",
    "--learning_rate", "2e-5",
    "--lr_scheduler_type", "linear",
    "--warmup_ratio", "0.03",
    "--weight_decay", "0.01",
    "--lora_rank", "16",
    "--lora_alpha", "32",
    "--freeze_vit", "true",
    "--gradient_checkpointing", "true",
    "--load_best_model_at_end", "true",
    "--metric_for_best_model", "eval_token_acc",
    "--greater_is_better", "true",
    "--save_steps", "1000",
    "--save_total_limit", "2",
    "--logging_steps", "20",
    "--output_dir", str(TRAIN_OUTPUT_DIR),
    "--report_to", "wandb" if os.getenv("WANDB_API_KEY") else "none",
    "--dataloader_num_workers", "2",
    "--dataset_num_proc", "2",
]

if os.getenv("HF_TOKEN"):
    swift_cmd += ["--use_hf", "true", "--push_to_hub", "true", "--hub_token", os.getenv("HF_TOKEN"), "--hub_model_id", HUB_MODEL_ID]

print("Training command:\n")
print(" ".join(shlex.quote(part) for part in swift_cmd))

DRY_RUN = True
if not DRY_RUN:
    subprocess.run(swift_cmd, check=True)

## 4. Deterministic Inference

Run deterministic inference for every test item and save one JSONL row per prediction. This file becomes the single source for answer evaluation, confidence analysis, and qualitative failure analysis.

In [ ]:
# Run in a GPU environment after training.
# !pip install ms-swift

PREDICTIONS_PATH = cfg.output_dir / "predictions.jsonl"
ADAPTER_PATH = os.getenv("PALIGEMMA_ADAPTER_PATH", "")  # local checkpoint or HF adapter repo

def load_eval_records(path: Path, sample_size: int | None = None, seed: int = SEED):
    rows = list(iter_jsonl(path))
    if sample_size is not None:
        rows = random.Random(seed).sample(rows, min(sample_size, len(rows)))
    return rows

eval_records = load_eval_records(cfg.test_jsonl, cfg.inference_sample_size)
print("Evaluation records:", len(eval_records))
print("Prediction output:", PREDICTIONS_PATH)

# Skeleton for real inference. Keep this cell explicit so model-loading failures are easy to debug.
RUN_INFERENCE = False

if RUN_INFERENCE:
    from swift.llm import PtEngine, RequestConfig, InferRequest
    import torch
    import gc

    gc.collect()
    torch.cuda.empty_cache()

    engine_kwargs = {
        "model_id_or_path": cfg.model_name,
        "max_batch_size": 2,
        "use_hf": True,
        "model_type": "paligemma",
    }
    if ADAPTER_PATH:
        engine_kwargs["adapters"] = ADAPTER_PATH
    engine = PtEngine(**engine_kwargs)
    request_config = RequestConfig(max_tokens=64, temperature=0)

    with PREDICTIONS_PATH.open("w", encoding="utf-8") as f:
        for start in range(0, len(eval_records), 16):
            batch = eval_records[start:start + 16]
            requests = []
            for record in batch:
                question = record["messages"][0]["content"].replace("<image>", "").strip()
                requests.append(InferRequest(messages=[{"role": "user", "content": f"<image>{question}"}], images=[record["images"][0]]))
            responses = engine.infer(requests, request_config)
            for record, response in zip(batch, responses):
                question = record["messages"][0]["content"].replace("<image>", "").strip()
                prediction = response.choices[0].message.content.strip()
                out = {
                    "question": question,
                    "reference": record["messages"][1]["content"],
                    "prediction": prediction,
                    "image": record["images"][0],
                    "metadata": record.get("metadata", {}),
                }
                f.write(json.dumps(out, ensure_ascii=False) + "\n")
            print(f"Processed {min(start + 16, len(eval_records))}/{len(eval_records)}")

## 5. Answer Evaluation

Use lexical metrics for continuity with benchmark reporting, but treat clinical correctness as the important endpoint. Start with exact/normalized matching, then add manual or clinician-assisted labels for a stratified subset.

In [ ]:
import re

def normalize_answer(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def exact_match(reference: str, prediction: str) -> bool:
    return normalize_answer(reference) == normalize_answer(prediction)

def evaluate_exact_match(predictions_path: Path) -> dict:
    rows = list(iter_jsonl(predictions_path))
    if not rows:
        return {"n": 0, "exact_match": None}
    matches = [exact_match(row["reference"], row["prediction"]) for row in rows]
    return {"n": len(rows), "exact_match": sum(matches) / len(matches)}

if PREDICTIONS_PATH.exists():
    print(json.dumps(evaluate_exact_match(PREDICTIONS_PATH), indent=2))
else:
    print("No predictions file yet. Run inference first.")

## 6. Visual Explanation Maps

Generate explanation maps for each prediction. Keep raw maps as arrays, not only PNGs, because perturbation tests need numeric saliency values.

Initial methods:

- Vision attention map
- Gradient/saliency map
- Later: token-region alignment for terms such as `polyp`, `ulcer`, `bleeding`, `lesion`

In [ ]:
EXPLANATION_DIR = cfg.output_dir / "explanation_maps"
EXPLANATION_DIR.mkdir(parents=True, exist_ok=True)

MEDICAL_TERMS = {
    "polyp", "polyps", "ulcer", "ulcers", "lesion", "lesions", "bleeding",
    "inflammation", "erythema", "erosion", "tumor", "mass", "nodule",
    "stricture", "diverticulum", "angiodysplasia", "abnormality", "abnormalities",
}

def extract_medical_terms(text: str) -> list[str]:
    lowered = normalize_answer(text)
    return sorted(term for term in MEDICAL_TERMS if term in lowered.split())

print("Explanation map directory:", EXPLANATION_DIR)
print("Medical terms tracked:", len(MEDICAL_TERMS))

## 7. Faithfulness Perturbation Tests

This is the main study contribution.

For each image/question pair:

1. Generate original answer and explanation map.
2. Mask top-k salient regions.
3. Mask random regions with the same area.
4. Mask least-salient regions.
5. Re-run inference.
6. Measure answer flip, exact-match change, and confidence change.

Faithful maps should show stronger degradation under targeted deletion than under controls.

In [ ]:
# Utilities for image perturbation. These can be used once explanation maps are saved as arrays.
# !pip install pillow numpy

from PIL import Image
import numpy as np

PERTURBED_DIR = cfg.output_dir / "perturbed_images"
PERTURBED_DIR.mkdir(parents=True, exist_ok=True)

def resize_saliency_to_image(saliency: np.ndarray, image_size: tuple[int, int]) -> np.ndarray:
    saliency_img = Image.fromarray((255 * saliency / (saliency.max() + 1e-8)).astype("uint8"))
    saliency_img = saliency_img.resize(image_size, resample=Image.Resampling.BILINEAR)
    arr = np.asarray(saliency_img).astype("float32")
    return arr / (arr.max() + 1e-8)

def make_deletion_mask(saliency: np.ndarray, fraction: float, mode: str, seed: int = SEED) -> np.ndarray:
    flat = saliency.reshape(-1)
    n_mask = max(1, int(len(flat) * fraction))
    if mode == "targeted":
        indices = np.argpartition(flat, -n_mask)[-n_mask:]
    elif mode == "least_salient":
        indices = np.argpartition(flat, n_mask)[:n_mask]
    elif mode == "random":
        rng = np.random.default_rng(seed)
        indices = rng.choice(len(flat), size=n_mask, replace=False)
    else:
        raise ValueError(f"Unknown deletion mode: {mode}")
    mask = np.zeros_like(flat, dtype=bool)
    mask[indices] = True
    return mask.reshape(saliency.shape)

def apply_gray_mask(image_path: str | Path, saliency: np.ndarray, fraction: float, mode: str, seed: int = SEED) -> Image.Image:
    image = Image.open(image_path).convert("RGB")
    image_arr = np.asarray(image).copy()
    saliency_resized = resize_saliency_to_image(saliency, image.size)
    mask = make_deletion_mask(saliency_resized, fraction=fraction, mode=mode, seed=seed)
    fill = np.array([127, 127, 127], dtype=np.uint8)
    image_arr[mask] = fill
    return Image.fromarray(image_arr)

print("Perturbation levels:", cfg.deletion_levels)
print("Perturbed image directory:", PERTURBED_DIR)

## 8. Bootstrap Confidence Intervals

Use paired bootstrap intervals over question-image pairs when comparing targeted vs random deletion.

In [ ]:
def bootstrap_ci(values, statistic_fn=statistics.mean, n_boot=2000, ci=0.95, seed=SEED):
    values = list(values)
    if not values:
        return {"estimate": None, "low": None, "high": None}
    rng = random.Random(seed)
    estimates = []
    for _ in range(n_boot):
        sample = [values[rng.randrange(len(values))] for _ in values]
        estimates.append(statistic_fn(sample))
    estimates.sort()
    alpha = 1 - ci
    low = estimates[int((alpha / 2) * n_boot)]
    high = estimates[int((1 - alpha / 2) * n_boot) - 1]
    return {"estimate": statistic_fn(values), "low": low, "high": high}

# Example once you have per-item paired differences:
# confidence_drop_difference = targeted_confidence_drops - random_confidence_drops
# bootstrap_ci(confidence_drop_difference)

## 9. Failure-Mode Taxonomy

Use this table while manually reviewing examples. The aim is to build a defensible taxonomy, not just cherry-pick interesting images.

| Category | Description |
| --- | --- |
| Correct and grounded | Answer correct; explanation highlights plausible visual evidence |
| Correct but ungrounded | Answer correct; deletion of highlighted region has little effect |
| Incorrect but plausible | Answer wrong; explanation looks clinically persuasive |
| Overconfident error | Answer wrong with high confidence |
| Visual misinterpretation | Model attends to visible region but labels it incorrectly |
| Context mismatch | Answer reflects question prior or language bias more than image evidence |
| Explanation-method disagreement | Attention and saliency point to different regions |

## 10. Reporting Checklist

- Dataset counts and split details
- Model, training, LoRA, quantization, and decoding settings
- Full-test VQA metrics
- Stratified metrics by question type if available
- Targeted vs random deletion degradation with bootstrap CIs
- Confidence calibration and overconfident-error analysis
- Representative qualitative examples from each failure category
- Limitations: no clinical deployment claim, explanation methods are approximations, dataset labels may be incomplete